# Model Training


In [73]:
# importing required data and packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#modelling
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error, mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings




In [74]:
# importing the csv file as dataframe
try:
    df = pd.read_csv("data/preprocessed/preprocessed.csv")
except:
    df = pd.read_csv("preprocessed.csv")



In [75]:
df = df.drop(columns=['id'], axis=1)

In [76]:
df.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk,speed_risk_band,high_curvature,lane_complexity,accident_density,poor_visibility
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13,low,0,0.12,0.333333,0
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35,low,1,3.96,0.000000,0
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30,medium,1,2.52,0.400000,0
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21,low,0,0.28,0.200000,0
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56,medium,0,0.58,0.500000,0


In [77]:
target = "accident_risk"

In [78]:
# --- PREPARING X AND Y VARIABLES---
X = df.drop(columns=[target], axis=1)
X.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,speed_risk_band,high_curvature,lane_complexity,accident_density,poor_visibility
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,low,0,0.12,0.333333,0
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,low,1,3.96,0.000000,0
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,medium,1,2.52,0.400000,0
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,low,0,0.28,0.200000,0
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,medium,0,0.58,0.500000,0


In [79]:
X.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,speed_risk_band,high_curvature,lane_complexity,accident_density,poor_visibility
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,low,0,0.12,0.333333,0
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,low,1,3.96,0.000000,0
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,medium,1,2.52,0.400000,0
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,low,0,0.28,0.200000,0
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,medium,0,0.58,0.500000,0


In [80]:
Y = df[target]
Y.head()

0    0.13
1    0.35
2    0.30
3    0.21
4    0.56
Name: accident_risk, dtype: float64

In [81]:
numerical_features = [
    "num_lanes",
    "curvature",
    "speed_limit",
    "lane_complexity"      # SAFE
]

categorical_features = [
    "road_type",
    "lighting",
    "weather",
    "time_of_day",
    "speed_risk_band"      # SAFE engineered category
]


boolean_features = [
    "road_signs_present",
    "public_road",
    "holiday",
    "school_season",
    "high_curvature",      # SAFE
    "poor_visibility"      # SAFE
]

In [82]:
df['school_season']

0          True
1          True
2         False
3         False
4         False
          ...  
517749    False
517750    False
517751     True
517752     True
517753     True
Name: school_season, Length: 517754, dtype: bool

In [83]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[

        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=False), categorical_features),
        ('bool', OneHotEncoder(handle_unknown='ignore',sparse_output=False), boolean_features)
    ]
)



In [84]:
preprocessor


ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['num_lanes', 'curvature', 'speed_limit',
                                  'lane_complexity']),
                                ('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['road_type', 'lighting', 'weather',
                                  'time_of_day', 'speed_risk_band']),
                                ('bool',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['road_signs_present', 'public_road',
                                  'holiday', 'school_season', 'high_curvature',
                                  'poor_visibility'])])

In [85]:

X = preprocessor.fit_transform(X)

In [86]:
X.shape

(517754, 30)

In [87]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print("x_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train_shape", y_train.shape)
print("y_test shape:", y_test.shape)

x_train shape: (414203, 30)
X_test shape: (103551, 30)
y_train_shape (414203,)
y_test shape: (103551,)


In [88]:
# create an evaluate function to evaluate models
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [89]:
models = {
    'linear_regression': LinearRegression(),
    'lasso_regression': Lasso(),
    'Ridge_regression': Ridge(),
    'KNN_regressor': KNeighborsRegressor(),
    'Decision_tree_regressor': DecisionTreeRegressor(),
    'Random_forest_regressor': RandomForestRegressor(),
    'XGB_regressor': XGBRegressor(),
    'cat_boost_regressor': CatBoostRegressor(),
    'Ada_boost_regressor': AdaBoostRegressor()
}

model_report = [] # to store model evaluation reports
r2_list = [] # to store r2 scores of different models

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    y_train_pred = model.predict(X_train)

    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_report.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

linear_regression
Model performance for Training set
- Root Mean Squared Error: 0.0627
- Mean Absolute Error: 0.0484
- R2 Score: 0.8583
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.0628
- Mean Absolute Error: 0.0485
- R2 Score: 0.8572


lasso_regression
Model performance for Training set
- Root Mean Squared Error: 0.1665
- Mean Absolute Error: 0.1330
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.1662
- Mean Absolute Error: 0.1327
- R2 Score: -0.0000


Ridge_regression
Model performance for Training set
- Root Mean Squared Error: 0.0627
- Mean Absolute Error: 0.0484
- R2 Score: 0.8583
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.0628
- Mean Absolute Error: 0.0485
- R2 Score: 0.8572


KNN_regressor
Model performance for Training set
- Root Mean Squared Error: 0.0568
- Mean Absolute Error: 0.0440
- R2 Score: 0.8835
--------

In [1]:
# result dataframe
result_df = pd.DataFrame({'Model':model_report, 'R2_score':r2_list, "rmse": })
result_df

SyntaxError: invalid syntax (1228500719.py, line 2)